# 02 - Data Quality

**Objectif :** appliquer les règles qualité et les diagnostics colonnes uniquement sur le train brut.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load raw train data

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

loader = CsvLoanDataLoader(path=settings.raw_train_path)
raw_train_df = loader.load()
raw_train_df.shape

## 2. Quality checker

In [ ]:
from credit_risk_lab.domain.entities import LoanSchema
from credit_risk_lab.infrastructure import CreditRiskQualityChecker

quality_checker = CreditRiskQualityChecker(schema=LoanSchema())
quality_report = quality_checker.validate(raw_train_df)
quality_report

## 3. Clean implausible rows

In [ ]:
clean_train_df = quality_checker.clean(raw_train_df)

pd.DataFrame(
    [
        {"dataset": "raw_train", "rows": len(raw_train_df), "duplicates": raw_train_df.duplicated().sum()},
        {"dataset": "clean_train", "rows": len(clean_train_df), "duplicates": clean_train_df.duplicated().sum()},
    ]
)

## 4. Inspect clean data

In [ ]:
from credit_risk_lab.infrastructure.analytics import DatasetInspector

clean_inspector = DatasetInspector(clean_train_df)
display(clean_inspector.summary())
clean_inspector.target_distribution(settings.target_column)

## 5. Numeric outlier diagnostics

In [ ]:
from credit_risk_lab.infrastructure.analytics import ColumnDiagnostics

diagnostics = ColumnDiagnostics(clean_train_df, target_column=settings.target_column)
numeric_outliers = diagnostics.numeric_outlier_report()
numeric_outliers

## 6. Numeric outlier visuals

In [ ]:
from credit_risk_lab.infrastructure.visualization import DataQualityVisualizer

visualizer = DataQualityVisualizer(
    frame=clean_train_df,
    target_column=settings.target_column,
)

visualizer.numeric_outlier_overview(
    numeric_outliers["feature"].head(6).tolist(),
).show()

## 7. Categorical diagnostics

In [ ]:
categorical_profile = diagnostics.categorical_profile(rare_threshold=0.01)
categorical_profile

## 8. Categorical visuals

In [ ]:
visualizer.categorical_feature_overview(
    categorical_profile["feature"].head(4).tolist(),
).show()